In [ ]:
# Cell 1: Install and Import
!pip install -q evaluate peft librosa soundfile audiomentations

import os
import torch
import random
import re
import pandas as pd
import numpy as np
import evaluate
import librosa
from tqdm import tqdm
from datasets import Dataset, Audio
from audiomentations import Compose, AddGaussianNoise, PitchShift
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# Enable tqdm for pandas
tqdm.pandas()

if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: GPU not detected! Make sure your kernel has CUDA enabled.")


In [ ]:
# Cell 2: Dataset Loading and Aggressive Cleaning
BASE_PATH = os.environ.get("MEETLY_DATASET_DIR")
if not BASE_PATH:
    raise RuntimeError("Set MEETLY_DATASET_DIR to the local dataset directory.")
csv_path = os.path.join(BASE_PATH, "augmented_combined_dataset.parquet")

df = pd.read_parquet(csv_path)

# Fix paths for Windows
df["audio"] = df["audio"].str.replace(r"\\", "/", regex=True)
df["audio"] = df["audio"].apply(lambda x: os.path.join(BASE_PATH, x).replace("\\", "/"))

# Filter out physical files that are missing
df = df[df["audio"].apply(os.path.exists)]
initial_rows = len(df)
print(f"Initial valid files found: {initial_rows}")

# 🚨 CLEANING LOGIC: Detect infinite repetition loops in Ground Truth
def has_repetition_loops(text, n=3, max_repeats=2):
    words = str(text).split()
    if len(words) < n * (max_repeats + 1): return False
    
    for i in range(len(words) - n * max_repeats):
        ngram = tuple(words[i:i+n])
        repeats = 1
        for j in range(i+n, len(words) - n + 1, n):
            if tuple(words[j:j+n]) == ngram:
                repeats += 1
                if repeats > max_repeats: return True
            else:
                break
    return False

# 🚨 CLEANING LOGIC: Detect Audio/Text length mismatches
def is_valid_sample(row):
    text = str(row["text"])
    audio_path = row["audio"]
    
    # 1. Reject if text contains Hallucination Loops (repeating 3 or 4 word phrases)
    if has_repetition_loops(text, n=3) or has_repetition_loops(text, n=4):
        return False
        
    # 2. Reject if extreme length mismatch
    try:
        duration = librosa.get_duration(path=audio_path)
    except:
        return False # Audio is broken
        
    num_words = len(text.split())
    if duration < 2.0 and num_words > 10: return False # Audio too short for text
    if duration > 10.0 and num_words < 2: return False # Audio too long for text
        
    return True

print("🧹 Scanning dataset for corrupted labels and mismatches (This may take a minute)...")
df["is_valid"] = df.progress_apply(is_valid_sample, axis=1)

df_clean = df[df["is_valid"] == True].copy()
removed = initial_rows - len(df_clean)

print(f"\n✅ Cleaning Complete!")
print(f"🗑️ Removed {removed} corrupted samples.")
print(f"🎯 Final High-Quality Dataset Size: {len(df_clean)} samples.")


In [ ]:
# Cell 3: Build Hugging Face Dataset & Augmentation
# 🚀 UPGRADE 1: Using the State-of-the-Art Large-V3 model
model_id = "openai/whisper-large-v3"

data_dict = {
    "path": df_clean["audio"].tolist(),
    "text": df_clean["text"].tolist(),
    "origin_source": df_clean["origin_source"].tolist()
}

dataset = Dataset.from_dict(data_dict)
dataset = dataset.cast_column("path", Audio(sampling_rate=16000))
dataset = dataset.train_test_split(test_size=0.15) 

print("✅ Train/Test Dataset created!")

processor = WhisperProcessor.from_pretrained(model_id, language="arabic", task="transcribe")

# 🚀 UPGRADE 2: Setup Audiomentations (Data Augmentation)
augmenter = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.3),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.2)
])

# 🚨 THE FIX: Separate Train and Test preparation!
def prepare_train(batch):
    audio = batch["path"]["array"]
    audio = augmenter(samples=audio, sample_rate=16000) # Noise ONLY for training
    batch["input_features"] = processor.feature_extractor(audio, sampling_rate=16000).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"], truncation=True, max_length=448).input_ids
    return batch

def prepare_test(batch):
    audio = batch["path"]["array"]
    # NO NOISE FOR TESTING!
    batch["input_features"] = processor.feature_extractor(audio, sampling_rate=16000).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"], truncation=True, max_length=448).input_ids
    return batch

print("⚙️ Processing Training Data (With Augmentations)...")
processed_train = dataset["train"].map(prepare_train, remove_columns=dataset["train"].column_names)

print("⚙️ Processing Test Data (Clean)...")
processed_test = dataset["test"].map(prepare_test, remove_columns=dataset["test"].column_names)

processed_dataset = {"train": processed_train, "test": processed_test}
print("✅ Features Extracted Perfectly!")


In [ ]:
# Cell 4: Data Collator and Metrics setup
from dataclasses import dataclass
from typing import Any

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        batch["input_features"] = batch["input_features"].to(torch.bfloat16)

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)

        # BOS Token fix
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer * 100}

print("✅ Collator and Metrics Ready.")


In [ ]:
# Cell 5: Setup LoRA
from peft import LoraConfig, get_peft_model

print("⏳ Loading Base Model...")
model = WhisperForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    # attn_implementation="flash_attention_2" # Uncomment if flash-attn installs successfully on your Windows machine!
)

model.config.use_cache = False
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="arabic", task="transcribe"
)
model.config.suppress_tokens = []

# 🚀 UPGRADE 3: Apply SpecAugment (Time and Frequency Masking)
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_feature_prob = 0.05

# 🚀 UPGRADE 4: Expand LoRA Target Modules
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"], # Targets all linear layers!
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, lora_config)

# This will prove LoRA is active by showing slightly higher trainable parameters (~1-2%)
model.print_trainable_parameters()


In [ ]:
# Cell 6: Optimized Training Arguments
from transformers import EarlyStoppingCallback

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-darja-lora-v2", 
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=8, # 🚀 ADDED: Speeds up testing significantly!
    
    learning_rate=1e-4, 
    weight_decay=0.01,
    
    lr_scheduler_type="cosine",
    warmup_steps=100,
    max_steps=1000,     
    
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,

    eval_strategy="steps", 
    eval_steps=300, # 🚀 CHANGED: Evaluates every 300 steps instead of 100
    save_steps=300, # 🚀 CHANGED: Saves every 300 steps to match
    save_total_limit=3,
    
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=225,
    
    label_names=["labels"] 
)

callbacks = [EarlyStoppingCallback(early_stopping_patience=4)]
print("✅ Optimized Training arguments configured.")


In [ ]:
# Load checkpoints using the installed PyTorch/Transformers defaults.

# Cell 7: Custom Logging, Visualizations, and Train!
import os
import pandas as pd
import matplotlib.pyplot as plt
from transformers import TrainerCallback

output_viz_dir = "Model_large_viz_train"
os.makedirs(output_viz_dir, exist_ok=True)
csv_log_path = os.path.join(output_viz_dir, "advanced_training_log.csv")

class AdvancedLoggingCallback(TrainerCallback):
    def __init__(self, log_path, viz_dir):
        self.log_path = log_path
        self.viz_dir = viz_dir
        self.logs = []
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
            
        current_log = {"step": state.global_step, "epoch": logs.get("epoch", None)}
        if "loss" in logs:
            current_log["train_loss"] = logs["loss"]
            current_log["learning_rate"] = logs.get("learning_rate", None)
        if "eval_loss" in logs:
            current_log["eval_loss"] = logs["eval_loss"]
        if "eval_wer" in logs:
            current_log["eval_wer"] = logs["eval_wer"]
            
        self.logs.append(current_log)
        
        df = pd.DataFrame(self.logs)
        df = df.groupby("step", as_index=False).first() 
        df.to_csv(self.log_path, index=False)
        self.generate_plots(df)
        
    def generate_plots(self, df):
        plt.figure(figsize=(10, 6))
        if "train_loss" in df.columns:
            train_df = df.dropna(subset=["train_loss"])
            plt.plot(train_df["step"], train_df["train_loss"], label="Train Loss", color="#1f77b4", linewidth=2)
            
        if "eval_loss" in df.columns:
            eval_df = df.dropna(subset=["eval_loss"])
            plt.plot(eval_df["step"], eval_df["eval_loss"], label="Eval Loss", color="#ff7f0e", marker="o", linewidth=2)
            
        plt.title("Training & Evaluation Loss", fontsize=14, fontweight='bold')
        plt.xlabel("Training Steps", fontsize=12)
        plt.ylabel("Loss", fontsize=12)
        plt.legend()
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.tight_layout()
        plt.savefig(os.path.join(self.viz_dir, "Loss_Curve.png"), dpi=300)
        plt.close()
        
        if "eval_wer" in df.columns:
            plt.figure(figsize=(10, 6))
            eval_df = df.dropna(subset=["eval_wer"])
            plt.plot(eval_df["step"], eval_df["eval_wer"], label="Eval WER (%)", color="#d62728", marker="s", linewidth=2)
            plt.title("Word Error Rate (WER) Over Time", fontsize=14, fontweight='bold')
            plt.xlabel("Training Steps", fontsize=12)
            plt.ylabel("WER (%)", fontsize=12)
            plt.legend()
            plt.grid(True, linestyle="--", alpha=0.6)
            plt.tight_layout()
            plt.savefig(os.path.join(self.viz_dir, "WER_Curve.png"), dpi=300)
            plt.close()

advanced_logger = AdvancedLoggingCallback(log_path=csv_log_path, viz_dir=output_viz_dir)
callbacks.append(advanced_logger)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks
)

print(f"\n--- 🟢 Resuming Training from Checkpoint (Fast Mode) ---")

# 🚀 CHANGED: This line finds your latest checkpoint and resumes automatically!
trainer.train(resume_from_checkpoint=True)

trainer.save_model("./whisper-darja-lora-v2")
processor.save_pretrained("./whisper-darja-lora-v2")
print("✅ Best LoRA adapter saved to ./whisper-darja-lora-v2")


In [ ]:
# Cell 8: Inference and Evaluation
import re
import random
import evaluate
import numpy as np
import pandas as pd
from tqdm import tqdm
from peft import PeftModel

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def normalize_text(text):
    text = re.sub(r'[^\w\s]', '', text)    
    text = re.sub(r'[أإآ]', 'ا', text)     
    text = re.sub(r'ة', 'ه', text)         
    text = re.sub(r'ى', 'ي', text)         
    text = " ".join(text.split())          
    return text

# 1. Load the original base model first
print("⏳ Loading Base Model...")
base_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-large-v3", # 🚀 MATCHES THE NEW UPGRADED BASE MODEL
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 2. Attach your new, cleanly trained LoRA adapter
saved_lora_path = "./whisper-darja-lora-v2" 
test_processor = WhisperProcessor.from_pretrained(saved_lora_path)
test_model = PeftModel.from_pretrained(base_model, saved_lora_path)
test_model.eval()

# 3. Setup Arabic Decoder
test_model.config.forced_decoder_ids = test_processor.get_decoder_prompt_ids(
    language="arabic", task="transcribe"
)
test_model.config.suppress_tokens = []
print("\n✅ New V2 LoRA Model loaded successfully!\n")

# 4. Evaluate on 100 random clean samples
num_samples = min(100, len(dataset["test"]))
random_indices = random.sample(range(len(dataset["test"])), num_samples)

results = []

print(f"🚀 Evaluating on {num_samples} random samples...")

for idx in tqdm(random_indices, desc="Evaluating"):
    sample = dataset["test"][idx]
    audio_path_str = sample["path"]["path"]
    
    input_features = test_processor.feature_extractor(
        sample["path"]["array"],
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(test_model.device).to(torch.bfloat16)

    with torch.no_grad():
        predicted_ids = test_model.generate(
            input_features,
            language="arabic",
            task="transcribe",
            num_beams=5,
            no_repeat_ngram_size=3,
            max_new_tokens=225,
        )

    pred_text = test_processor.tokenizer.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    norm_truth = normalize_text(sample["text"])
    norm_pred = normalize_text(pred_text)

    if len(norm_truth) > 0:
        w_score = wer_metric.compute(predictions=[norm_pred], references=[norm_truth]) * 100
        c_score = cer_metric.compute(predictions=[norm_pred], references=[norm_truth]) * 100
    else:
        w_score, c_score = 0.0, 0.0
        
    results.append({
        "audio_path": audio_path_str,
        "ground_truth": norm_truth,
        "prediction": norm_pred,
        "wer": w_score,
        "cer": c_score
    })

# 5. Save results and print metrics
df = pd.DataFrame(results)
csv_filename = "v2_evaluation_results.csv"
df.to_csv(csv_filename, index=False, encoding="utf-8-sig")

print(f"\n✅ All results successfully saved to {csv_filename}")
print("\n--- 📊 V2 Evaluation Metrics (100 Samples) ---")
print(f"Average WER: {df['wer'].mean():.2f}%")
print(f"Average CER: {df['cer'].mean():.2f}%")
